# QC: Data Collate Pipeline — SmolVLM2 vs Qwen3-SmVL

Apple-to-apple comparison of the two preprocessing pipelines on **DanQing** samples (`source="danqing"`).

| Dimension | SmolVLM2 (`dataset.py`) | Qwen3-SmVL (`train.py`) |
|---|---|---|
| Tokenizer | SmolVLM2 / Idefics | Qwen3 |
| Image token | `<image>` | `<\|image_pad\|>` |
| Global image token | `<global-img>` | `<\|vision_pad\|>` |
| Turn delimiter | `<end_of_utterance>` | `<\|im_end\|>` |
| System marker | `System:` | `<\|im_start\|>system\n` |
| User marker | `User:` | `<\|im_start\|>user\n` |
| System prompt lang | English | Chinese |
| End-marker in mask? | No (exclusive) | No (exclusive) |
| default mask_system | True | True |
| default mask_user | False | False |

In [ ]:
import sys, os

QWEN_ROOT = r"D:\cursorprojects\Qwen3-SmVL"
SMOL_ROOT = r"D:\cursorprojects\smolvlm2"

for p in [QWEN_ROOT, SMOL_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# train.py reads chat_template.jinja and model/ as relative paths
os.chdir(QWEN_ROOT)
print("cwd  :", os.getcwd())
print("paths:", [p for p in sys.path if "cursorprojects" in p])

In [ ]:
## 1 — Load DanQing samples

In [ ]:
from train import load_mixed_data_v2

DANQING_PATH = r"D:\cursorprojects\Qwen3-SmVL\data\DanQing\data"
N_SAMPLES    = 5

ds = load_mixed_data_v2(
    [{"path": DANQING_PATH, "label": "DanQing", "source": "danqing", "count": N_SAMPLES}],
    seed=42,
)
print(f"Loaded {len(ds)} samples  |  columns: {ds.column_names}")

ex0 = ds[0]
print("\nSample 0 — texts :", ex0["texts"])
print("Sample 0 — image  :", type(ex0["images"][0]))

In [ ]:
## 2 — Load processors

In [ ]:
# ── Qwen3-SmVL processor ──────────────────────────────────────────────────────
# Spliced: SmolVLM2 image processor + Qwen3 tokenizer, configured in utils.py
import utils as qwen_utils

try:
    qwen_processor = qwen_utils.load_processor()
    print("Qwen3-SmVL processor OK")
    print("  image_token       :", qwen_processor.image_token)
    print("  global_image_token:", qwen_processor.global_image_token)
    print("  fake_image_token  :", qwen_processor.fake_image_token)
    print("  tokenizer vocab   :", qwen_processor.tokenizer.vocab_size)
except Exception as e:
    qwen_processor = None
    print("[SKIP] Qwen3-SmVL processor:", e)

In [ ]:
# ── SmolVLM2 original processor (from HuggingFace Hub) ───────────────────────
from transformers import AutoProcessor

try:
    smol_processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-256M-Instruct")
    print("SmolVLM2 processor OK")
    print("  model_max_length:", smol_processor.tokenizer.model_max_length)
    print("  vocab size      :", smol_processor.tokenizer.vocab_size)
    # Inspect key special tokens
    smol_vocab = smol_processor.tokenizer.get_vocab()
    for tok in ["<image>", "<global-img>", "<end_of_utterance>"]:
        print(f"  {tok!r:<28} id={smol_vocab.get(tok, 'NOT FOUND')}")
except Exception as e:
    smol_processor = None
    print("[SKIP] SmolVLM2 processor:", e)

## 3 — SmolVLM2 pipeline helpers

Ported verbatim from `smolvlm2/smolvlm/datasets/dataset.py`:
- `_search_subsequence` → `_smol_find`
- `SupervisedDataset._mask_special_tokens` → `_smol_mask_special`
- `_mask_system_tokens` → `_smol_mask_system`
- `_mask_user_tokens` → `_smol_mask_user`
- `SupervisedDataset._get_item` (image path only) → `run_smol_pipeline`

In [ ]:
import re, torch

_SMOL_IGN    = -100
_SMOL_SYS    = ("You are a helpful language and vision assistant. "
                "You are able to understand the visual content that the user provides, "
                "and assist the user with a variety of tasks using natural language.")
_SMOL_INTRO  = "Here are some images: "
_SMOL_OUTTRO = "Now answer the following question: "


def _smol_find(seq: torch.Tensor, pattern: list, start: int = 0) -> int:
    """_search_subsequence from dataset.py — verbatim."""
    sl, pl = seq.tolist(), len(pattern)
    if pl == 0:
        return -1
    for i in range(start, len(sl) - pl + 1):
        if sl[i:i+pl] == pattern:
            return i
    return -1


def _smol_mask_special(ids: torch.Tensor, lbl: torch.Tensor, processor) -> None:
    """SupervisedDataset._mask_special_tokens — verbatim."""
    tok = processor.tokenizer
    lbl[ids == tok.pad_token_id] = _SMOL_IGN
    if hasattr(tok, "additional_special_tokens") and "<image>" in tok.additional_special_tokens:
        lbl[ids == tok.convert_tokens_to_ids("<image>")] = _SMOL_IGN
    if "<global-img>" in tok.get_vocab():
        lbl[ids == tok.convert_tokens_to_ids("<global-img>")] = _SMOL_IGN
    pat = re.compile(r"<row_\d+_col_\d+>")
    for t in (v for v in tok.get_vocab() if pat.fullmatch(v)):
        lbl[ids == tok.convert_tokens_to_ids(t)] = _SMOL_IGN


def _smol_mask_system(ids: torch.Tensor, lbl: torch.Tensor, tokenizer) -> None:
    """_mask_system_tokens from dataset.py — verbatim.
    Masks: System: ... <end_of_utterance>  (end token itself NOT masked)."""
    sys_ids = tokenizer.encode("System:", add_special_tokens=False)
    end_ids = tokenizer.encode("<end_of_utterance>", add_special_tokens=False)
    pos = 0
    while True:
        s = _smol_find(ids, sys_ids, pos)
        if s == -1: break
        e = _smol_find(ids, end_ids, s + len(sys_ids))
        lbl[s : (len(ids) if e == -1 else e)] = _SMOL_IGN
        pos = (len(ids) if e == -1 else e) + len(end_ids)


def _smol_mask_user(ids: torch.Tensor, lbl: torch.Tensor, tokenizer) -> None:
    """_mask_user_tokens from dataset.py — verbatim.
    Masks: User: ... <end_of_utterance>  (end token itself NOT masked)."""
    usr_ids = tokenizer.encode("User:", add_special_tokens=False)
    end_ids = tokenizer.encode("<end_of_utterance>", add_special_tokens=False)
    pos = 0
    while True:
        s = _smol_find(ids, usr_ids, pos)
        if s == -1: break
        e = _smol_find(ids, end_ids, s + len(usr_ids))
        lbl[s : (len(ids) if e == -1 else e)] = _SMOL_IGN
        pos = (len(ids) if e == -1 else e) + len(end_ids)


def run_smol_pipeline(
    example: dict,
    processor,
    add_media_intro_outro: bool = False,
    mask_system_tokens:    bool = True,
    mask_user_tokens:      bool = False,
) -> dict | None:
    """
    Mirrors SupervisedDataset._get_item() (image-only path) from dataset.py.
    Input: single DanQing example  {"images": [...], "texts": [{"user":…,"assistant":…}]}.
    Returns: {"formatted_text", "input_ids", "labels", "active"}.
    """
    if processor is None:
        return None

    images = example["images"]
    if not isinstance(images, list): images = [images]
    images = images[:1]

    turn = example["texts"]
    if isinstance(turn, list): turn = turn[0]

    messages = [
        {"role": "system",    "content": [{"type": "text",  "text": _SMOL_SYS}]},
        {"role": "user",      "content": [{"type": "image"}] * len(images)
                                        + [{"type": "text", "text": turn["user"]}]},
        {"role": "assistant", "content": [{"type": "text",  "text": turn["assistant"]}]},
    ]

    if add_media_intro_outro:
        uc = messages[1]["content"]
        last_img = max((i for i,c in enumerate(uc) if c.get("type")=="image"), default=None)
        if last_img is not None:
            uc.insert(0, {"type": "text", "text": _SMOL_INTRO})
            uc.insert(last_img + 2, {"type": "text", "text": _SMOL_OUTTRO})

    fmt = processor.apply_chat_template(messages, add_generation_prompt=False)
    enc = processor(text=fmt, images=images, return_tensors="pt", padding=False)

    ids = enc["input_ids"][0]
    lbl = ids.clone()
    _smol_mask_special(ids, lbl, processor)
    if mask_system_tokens: _smol_mask_system(ids, lbl, processor.tokenizer)
    if mask_user_tokens:   _smol_mask_user(ids, lbl, processor.tokenizer)

    return {"formatted_text": fmt, "input_ids": ids, "labels": lbl,
            "active": (lbl != _SMOL_IGN)}

print("SmolVLM2 helpers ready")

## 4 — Qwen3-SmVL pipeline helper

Single-sample, no-device-move version of `data_collate_fix2k` from `train.py`.
Imports the masking functions directly so the logic is identical.

In [ ]:
from train import (
    IGNORE_INDEX,
    _inject_system_message,
    _inject_image_intro_outro,
    _mask_special_tokens,
    _mask_system_tokens,
    _mask_user_tokens,
)

_QWEN_SYS    = "你是一个有帮助的语言与视觉助手。你能够理解用户提供的视觉内容，并使用自然语言协助用户完成各种任务。"
_QWEN_INTRO  = "以下是一些图片："
_QWEN_OUTTRO = "现在请回答以下问题："


def run_qwen_pipeline(
    example: dict,
    processor,
    add_media_intro_outro: bool = False,
    mask_system_tokens:    bool = True,
    mask_user_tokens:      bool = False,
    system_message:        str  = None,
) -> dict | None:
    """
    Single-sample version of data_collate_fix2k from train.py.
    Input: single DanQing example  {"images": [...], "texts": [{"user":…,"assistant":…}]}.
    Returns: {"formatted_text", "input_ids", "labels", "active"}.
    """
    if processor is None:
        return None

    sys_msg = system_message or _QWEN_SYS
    images  = example["images"]
    if not isinstance(images, list): images = [images]
    images = images[:1]

    turn = example["texts"]
    if isinstance(turn, list): turn = turn[0]

    messages = [
        {"role": "user",
         "content": [{"type": "image"}] * len(images)
                   + [{"type": "text", "text": turn["user"]}]},
        {"role": "assistant",
         "content": [{"type": "text", "text": turn["assistant"]}]},
    ]

    # Step 1: inject system prompt
    messages = _inject_system_message(messages, sys_msg)
    # Step 2 (optional): inject image intro / outro
    if add_media_intro_outro:
        messages = _inject_image_intro_outro(messages, _QWEN_INTRO, _QWEN_OUTTRO)

    fmt = processor.apply_chat_template(
        messages, enable_thinking=False, add_generation_prompt=False
    )

    # processor expects images as list-of-lists (batch dim = 1)
    enc = processor(
        text=fmt, images=[images],
        return_tensors="pt", padding=False,
        truncation=True, max_length=4096,
    )

    ids = enc["input_ids"][0]
    lbl = ids.clone()
    # Layer 1: always — pad + vision special tokens
    _mask_special_tokens(ids, lbl, processor)
    # Layer 2: system turn  <|im_start|>system\n … <|im_end|>  (end NOT masked)
    if mask_system_tokens: _mask_system_tokens(ids, lbl, processor.tokenizer)
    # Layer 3: user turn    <|im_start|>user\n   … <|im_end|>  (end NOT masked)
    if mask_user_tokens:   _mask_user_tokens(ids, lbl, processor.tokenizer)

    return {"formatted_text": fmt, "input_ids": ids, "labels": lbl,
            "active": (lbl != IGNORE_INDEX)}

print("Qwen3-SmVL helper ready")

## 5 — Run both pipelines on all samples

In [ ]:
results = []

for idx in range(len(ds)):
    ex   = ds[idx]
    smol = run_smol_pipeline(ex, smol_processor)
    qwen = run_qwen_pipeline(ex, qwen_processor)
    results.append({"smol": smol, "qwen": qwen})

    s_tok = len(smol["input_ids"])          if smol else "N/A"
    q_tok = len(qwen["input_ids"])          if qwen else "N/A"
    s_act = int(smol["active"].sum())       if smol else "N/A"
    q_act = int(qwen["active"].sum())       if qwen else "N/A"
    print(f"[{idx}] SmolVLM2  tokens={s_tok:>5}  active={s_act:>5}  "
          f"| Qwen3-SmVL  tokens={q_tok:>5}  active={q_act:>5}")

print("\nAll done.")

## 6 — Raw formatted-text comparison

Set `SHOW_IDX` to switch between samples.

In [ ]:
import re as _re

SHOW_IDX = 0   # ← change to 0..4 to inspect different samples

def compact(t: str, max_chars: int = 2500) -> str:
    """Replace long runs of image tokens with a short placeholder."""
    t = _re.sub(r'(<image>\n?){2,}',     lambda m: f'<image×{m.group().count("<image>")}>\n', t)
    t = _re.sub(r'(<\|image_pad\|>\n?){2,}', lambda m: f'<pad×{m.group().count("<|image_pad|>")}>\n', t)
    t = _re.sub(r'(<\|vision_pad\|>)',   '<vision_pad>', t)
    return t[:max_chars]

r = results[SHOW_IDX]
smol_txt = r["smol"]["formatted_text"] if r["smol"] else "(unavailable)"
qwen_txt = r["qwen"]["formatted_text"] if r["qwen"] else "(unavailable)"

SEP = "═" * 72
print(SEP)
print(f"[Sample {SHOW_IDX}]  SmolVLM2 — formatted text")
print(SEP)
print(compact(smol_txt))
print()
print(SEP)
print(f"[Sample {SHOW_IDX}]  Qwen3-SmVL — formatted text")
print(SEP)
print(compact(qwen_txt))

## 7 — Unified diff of formatted text

Image token runs are collapsed before diffing so structural differences stand out clearly.

In [ ]:
import difflib

def norm_for_diff(t: str) -> str:
    """Collapse repeated image tokens and normalise whitespace for clean diffing."""
    t = _re.sub(r'(<image>\n?)+',         '<IMAGE_TOKENS>\n', t)
    t = _re.sub(r'(<\|image_pad\|>\n?)+', '<IMAGE_TOKENS>\n', t)
    t = _re.sub(r'(<\|vision_pad\|>)+',   '<GLOBAL_IMG>',     t)
    t = _re.sub(r'(<global-img>)+',       '<GLOBAL_IMG>',     t)
    return t

r = results[SHOW_IDX]
a = norm_for_diff(r["smol"]["formatted_text"] if r["smol"] else "")
b = norm_for_diff(r["qwen"]["formatted_text"] if r["qwen"] else "")

diff = list(difflib.unified_diff(
    a.splitlines(keepends=True),
    b.splitlines(keepends=True),
    fromfile="SmolVLM2", tofile="Qwen3-SmVL", n=3,
))
print("".join(diff[:150]) if diff else "(no structural differences after normalisation)")

## 8 - Token-level masking visualisation (HTML)

Green = active (contributes to loss).  Red = masked (IGNORE_INDEX).

In [ ]:
from IPython.display import display, HTML
import html as _html

def render_mask_html(result: dict, tokenizer, title: str, max_tokens: int = 350) -> str:
    if result is None:
        return f"<p><b>{_html.escape(title)}</b> - unavailable</p>"

    ids    = result["input_ids"][:max_tokens]
    active = result["active"][:max_tokens]
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())

    spans = []
    for tok, act in zip(tokens, active.tolist()):
        tok_str = _html.escape((tok or "<UNK>").replace("▁", "_"))
        bg  = "#c8e6c9" if act else "#ffcdd2"
        bdr = "#388e3c" if act else "#c62828"
        spans.append(
            f'<span style="background:{bg};border:1px solid {bdr};'
            f'border-radius:3px;padding:1px 4px;margin:1px;'
            f'font-family:monospace;font-size:11px;display:inline-block">'
            f'{tok_str}</span>'
        )

    n_total  = len(result["input_ids"])
    n_active = int(result["active"].sum())
    n_masked = n_total - n_active
    stats = (
        f"<small>total={n_total} | "
        f"<span style='color:#388e3c'>active={n_active}</span> | "
        f"<span style='color:#c62828'>masked={n_masked}</span> | "
        f"active%={100*n_active/n_total:.1f}%</small>"
    )
    return (
        f"<div style='border:2px solid #888;border-radius:6px;"
        f"padding:10px;margin:10px 0'>"
        f"<b>{_html.escape(title)}</b>&nbsp;&nbsp;{stats}"
        f"<div style='margin-top:8px;line-height:2.2'>{''.join(spans)}</div>"
        f"<p style='color:#aaa;font-size:10px;margin-top:4px'>"
        f"first {max_tokens} of {n_total} tokens shown</p>"
        f"</div>"
    )

r = results[SHOW_IDX]

smol_html = render_mask_html(
    r["smol"],
    smol_processor.tokenizer if smol_processor else None,
    f"SmolVLM2 - sample {SHOW_IDX}  (mask_system=True, mask_user=False)",
) if smol_processor else "<p>SmolVLM2 processor unavailable</p>"

qwen_html = render_mask_html(
    r["qwen"],
    qwen_processor.tokenizer if qwen_processor else None,
    f"Qwen3-SmVL - sample {SHOW_IDX}  (mask_system=True, mask_user=False)",
) if qwen_processor else "<p>Qwen3-SmVL processor unavailable</p>"

display(HTML(smol_html + qwen_html))

## 9 - Effect of add_media_intro_outro

In [ ]:
ex = ds[SHOW_IDX]

s_no  = run_smol_pipeline(ex, smol_processor, add_media_intro_outro=False)
s_yes = run_smol_pipeline(ex, smol_processor, add_media_intro_outro=True)
q_no  = run_qwen_pipeline(ex, qwen_processor, add_media_intro_outro=False)
q_yes = run_qwen_pipeline(ex, qwen_processor, add_media_intro_outro=True)

SEP = "-" * 60
for label, out in [
    ("SmolVLM2  WITHOUT intro/outro", s_no),
    ("SmolVLM2  WITH    intro/outro", s_yes),
    ("Qwen3-SmVL  WITHOUT intro/outro", q_no),
    ("Qwen3-SmVL  WITH    intro/outro", q_yes),
]:
    print(SEP)
    print(label)
    print(SEP)
    print(compact(out["formatted_text"]) if out else "N/A")
    print()

## 10 - Masking config sweep  (mask_system x mask_user)

In [ ]:
ex = ds[SHOW_IDX]
CONFIGS = [
    (True,  False, "default   sys=Y usr=N"),
    (False, False, "no-mask   sys=N usr=N"),
    (True,  True,  "full-mask sys=Y usr=Y"),
    (False, True,  "usr-only  sys=N usr=Y"),
]

hdr = f"{'Config':<26} | {'SmolVLM2':^30} | {'Qwen3-SmVL':^30}"
print(hdr)
print("-" * len(hdr))
for ms, mu, lbl in CONFIGS:
    s = run_smol_pipeline(ex, smol_processor, mask_system_tokens=ms, mask_user_tokens=mu)
    q = run_qwen_pipeline(ex, qwen_processor, mask_system_tokens=ms, mask_user_tokens=mu)

    def fmt(out):
        if out is None: return "N/A"
        n = len(out["input_ids"]); a = int(out["active"].sum())
        return f"total={n}  active={a}  ({100*a/n:.0f}%)"

    print(f"{lbl:<26} | {fmt(s):^30} | {fmt(q):^30}")

## 11 - Summary stats table (all 5 samples)

In [ ]:
import pandas as pd

rows = []
for idx, r in enumerate(results):
    q_txt = r["qwen"]["formatted_text"] if r["qwen"] else ""
    s_txt = r["smol"]["formatted_text"] if r["smol"] else ""
    for key, label in [("smol", "SmolVLM2"), ("qwen", "Qwen3-SmVL")]:
        out = r[key]
        if out is None:
            rows.append({"sample": idx, "model": label,
                         "total": None, "active": None, "masked": None, "active%": None,
                         "text_chars": None})
            continue
        n = len(out["input_ids"])
        a = int(out["active"].sum())
        rows.append({
            "sample":    idx,
            "model":     label,
            "total":     n,
            "active":    a,
            "masked":    n - a,
            "active%":   f"{100*a/n:.1f}%",
            "text_chars": len(out["formatted_text"]),
        })

df = pd.DataFrame(rows)
df.set_index(["sample", "model"], inplace=True)
df

## 12 - HTML masking for all 5 samples

Scroll through to compare masking patterns across different Q&A pairs.

In [ ]:
html_parts = []
for idx, r in enumerate(results):
    html_parts.append(
        f"<h3 style='margin-top:20px;border-bottom:1px solid #ccc'>"
        f"Sample {idx} &mdash; user: "
        f"{_html.escape(str(ds[idx]['texts'][0]['user'])[:80])}..."
        f"</h3>"
    )
    html_parts.append(render_mask_html(
        r["smol"],
        smol_processor.tokenizer if smol_processor else None,
        f"SmolVLM2",
        max_tokens=300,
    ) if smol_processor else "<p>SmolVLM2 unavailable</p>")
    html_parts.append(render_mask_html(
        r["qwen"],
        qwen_processor.tokenizer if qwen_processor else None,
        f"Qwen3-SmVL",
        max_tokens=300,
    ) if qwen_processor else "<p>Qwen3-SmVL unavailable</p>")

display(HTML("\n".join(html_parts)))